## Welcome to the Second Lab - Week 1, Day 3

Today we will work with lots of models! This is a way to get comfortable with APIs.

In [ ]:
# Start with imports - ask ChatGPT to explain any package that you don't know
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [ ]:
# Always remember to do this!
load_dotenv(override=True)

In [ ]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

In [ ]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [ ]:
messages

In [ ]:
openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=messages,
)
question = response.choices[0].message.content
print(question)

In [ ]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

In [ ]:
# The API we know well
# I've updated this with the latest model, but it can take some time because it likes to think!
# Replace the model with gpt-4.1-mini if you'd prefer not to wait 1-2 mins

model_name = "gpt-5-nano"

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-2.5-flash"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
model_name = "gemini-3-flash-preview"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# So where are we?
print(competitors)
print(answers)

In [ ]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


In [ ]:
# Let's bring this together - note the use of "enumerate"
together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [ ]:
print(together)

In [ ]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [ ]:
print(judge)

In [ ]:
judge_messages = [{"role": "user", "content": judge}]

In [ ]:
# Judgement time!
openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=judge_messages,
)
results = response.choices[0].message.content
print(results)


In [ ]:
# OK let's turn this into results!
results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

### Pattern(s) already used in this notebook

| Pattern | Where it appears |
|---|---|
| **Multi-Agent Collaboration** | Multiple LLMs (GPT, Gemini Flash, Gemini Pro) independently answer the same question |
| **LLM-as-a-Judge / Orchestration** | A separate GPT instance acts as an orchestrator: it generates the question, collects all responses, then evaluates and ranks them |

Together these form the **"parallel generation + judge"** agentic workflow.

---

### New pattern being added below: **Reflection**

The Reflection pattern adds a *feedback loop*:
1. **Critique** — the judge analyses *why* the worst answer lost
2. **Reflect & Revise** — the losing model sees the critique and rewrites its answer
3. **Re-judge** — the revised answer is compared against the original winner

This loop can be iterated until quality converges.

In [ ]:
# ============================================================
# REFLECTION PATTERN
# ============================================================
# Pattern summary:
#   1. CRITIQUE   — Ask the judge WHY the last-place competitor lost
#                   and what specifically was weak in its response.
#   2. REFLECT    — Feed that critique back to the losing competitor
#                   so it can revise its answer.
#   3. RE-JUDGE   — Compare the revised answer against the original
#                   winner to see whether quality improved.
#
# This closes the "generate → evaluate → improve" loop, which is
# the defining characteristic of the Reflection agentic pattern.
# ============================================================

import json
from openai import OpenAI

openai_client = OpenAI()

# ----------------------------------------------------------
# Step 0: Identify the loser from the previous judge ranking
# ----------------------------------------------------------
# 'ranks'       — list of competitor numbers ordered best→worst (from previous cells)
# 'competitors' — list of model names in the same positional order
# 'answers'     — list of model answers in the same positional order
# 'question'    — the original question every competitor answered

# The last element in `ranks` is the worst-ranked competitor number (1-based string)
loser_rank_number = ranks[-1]                              # e.g. "3"
loser_index       = int(loser_rank_number) - 1            # convert to 0-based index
loser_model       = competitors[loser_index]
loser_answer      = answers[loser_index]

winner_rank_number = ranks[0]
winner_index       = int(winner_rank_number) - 1
winner_model       = competitors[winner_index]
winner_answer      = answers[winner_index]

print(f"Winner : {winner_model}")
print(f"Loser  : {loser_model}")

# ----------------------------------------------------------
# Step 1: CRITIQUE — ask the judge to explain the loser's flaws
# ----------------------------------------------------------
critique_prompt = f"""You previously judged responses to this question:

{question}

The weakest response was from Competitor {loser_rank_number}:

{loser_answer}

Please provide specific, constructive critique. Explain exactly what was unclear,
missing, or logically weak. Be direct so the model can act on your feedback."""

critique_messages = [{"role": "user", "content": critique_prompt}]

critique_response = openai_client.chat.completions.create(
    model="gpt-4.1-mini",          # Judge model — can use any capable LLM
    messages=critique_messages,
)
critique = critique_response.choices[0].message.content
print("\n--- CRITIQUE ---\n", critique)

# ----------------------------------------------------------
# Step 2: REFLECT — send the critique back to the losing model
# so it can revise its answer (Reflection loop)
# ----------------------------------------------------------
# We re-use whichever client matches the losing competitor.
# For simplicity we route through the gemini client if it is a
# Gemini model, otherwise fall back to the OpenAI-compatible client.

if "gemini" in loser_model.lower():
    reflect_client = OpenAI(
        api_key=google_api_key,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )
else:
    reflect_client = openai_client          # Works for any OpenAI model

reflect_prompt = f"""You previously answered this question:

{question}

Your original answer was:

{loser_answer}

A judge reviewed your answer and gave the following critique:

{critique}

Please reflect on this critique and write an improved answer.
Focus on addressing every point raised."""

reflect_messages = [{"role": "user", "content": reflect_prompt}]

reflect_response = reflect_client.chat.completions.create(
    model=loser_model,             # Same model attempts to self-improve
    messages=reflect_messages,
)
revised_answer = reflect_response.choices[0].message.content
print("\n--- REVISED ANSWER ---\n", revised_answer)

# ----------------------------------------------------------
# Step 3: RE-JUDGE — compare the revised answer against the
# original winner to see whether the Reflection loop helped
# ----------------------------------------------------------
rejudge_prompt = f"""You are evaluating two responses to this question:

{question}

Response A (original winner, from {winner_model}):
{winner_answer}

Response B (revised answer, from {loser_model} after reflection):
{revised_answer}

Evaluate which response is better: clearer, more accurate, and more insightful.
Respond with JSON only, in this format:
{{"winner": "A or B", "reason": "one sentence explanation"}}"""

rejudge_messages = [{"role": "user", "content": rejudge_prompt}]

rejudge_response = openai_client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=rejudge_messages,
)
rejudge_result = json.loads(rejudge_response.choices[0].message.content)

print("\n--- RE-JUDGE RESULT ---")
print(f"Winner after Reflection loop: Response {rejudge_result['winner']}")
print(f"Reason: {rejudge_result['reason']}")

if rejudge_result["winner"] == "B":
    print(f"\n✓ Reflection worked — {loser_model} improved enough to beat {winner_model}!")
else:
    print(f"\n✗ Reflection did not flip the result — {winner_model} still leads.")